# FPUNet: retrain and test (team notebook)

FPUNet removes furniture and clutter from 2D occupancy-grid maps. Code and original model: **P2 Explore** (Song et al., IROS 2025), github.com/song-kun/P2Explore. Cite their paper if you use it.

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save.*
Run the cells **one at a time, from top to bottom** (not "Run all", because some cells ask you to choose a file). Keep this tab open while it trains.

**Files you need** (from our GitHub repo):

| File | Used in |
|---|---|
| `ipa_training_maps.zip` | **3-0**: 15 extra training buildings |
| `fpunet_test_maps.zip` | **5a**: 5 test buildings + 4 real robot maps, never used for training |
| `real_maps_for_colab.zip` | **Part 2** (optional): real robot maps as .pgm + .yaml |
| `best_fpunet_128_10.pth` | **5b** (optional): our first retrained model, shared by Google Drive |

### To retrain FPUNet (20 epochs, wall weight 0.6)
**Part 1 → 3-0 → 3a → 3b → 3c → 3d**, then **Part 5** to compare.
3a takes 15–50 min; training takes about 5.5 min per epoch (about 2 hours for 20 epochs). You need about **2 GB free** in your Google Drive.

### If Colab disconnects
Nothing is lost: everything is saved in your Drive. Reconnect, run **Part 1**, then **3R**, then **3c** and **3d** again. Training continues from the last finished epoch.

All results are saved in your Drive in **`FPUNet_models/w06_e20_loss`**.

## Part 1: Setup
Run these every time you open the notebook or reconnect.

In [ ]:
# 1a. Check the GPU (should print a GPU name like 'Tesla T4')
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU! Go to Runtime > Change runtime type > T4 GPU > Save, then run this cell again.")

In [ ]:
# 1b. Get the P2 Explore code and its maps (map.zip is inside the repo)
%cd /content
![ -d P2Explore ] || git clone -q https://github.com/song-kun/P2Explore.git
%cd /content/P2Explore
!cd dataset && unzip -q -o map.zip && ls
!pip -q install sortedcontainers gdown
!mkdir -p trained_model dataset/gen_data

In [ ]:
# 1c. Download the authors' trained model (about 80 MB) from their Google Drive link
!gdown 1v0JAYG_m0ZG8ol_Em5d-EqevraClvgx3 -O trained_model/best_fpunet_128.pth
import os
ok = os.path.exists("trained_model/best_fpunet_128.pth") and os.path.getsize("trained_model/best_fpunet_128.pth") > 1e6
print("model ready" if ok else "download failed: upload best_fpunet_128.pth into the trained_model folder (Files panel on the left)")

In [ ]:
# 1d. Connect your Google Drive (models and the training data backup are saved there)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/FPUNet_models
!ls /content/drive/MyDrive/FPUNet_models

## Part 2 (optional): Clean a real map and see before / after
**Skip this part when you are retraining.**

Run **2a** and choose **both** files of a map: the **.pgm** and the **.yaml** (Ctrl+click), for example from `real_maps_for_colab.zip` or from SLAM Toolbox's *Save Map*. Then run **2b**.
The map is shrunk to 0.2 m cells (what FPUNet works on), cleaned in 128 × 128 pieces, and FPUNet's answer is kept only inside the scanned area. Red = what FPUNet erased.

2b uses the new 20-epoch model if you have trained it, otherwise the authors' original model. To use another model, paste its path into `CKPT` at the top of 2b. For the next map, run 2a again, then 2b.

In [ ]:
# 2a. Upload a map: choose BOTH the .pgm and the .yaml
from google.colab import files
uploaded = files.upload()
YAML = [n for n in uploaded if n.endswith(".yaml")][0]
print("using", YAML)

In [ ]:
# 2b. Clean the map with FPUNet and show before / after
import os, sys, types
import numpy as np, cv2, torch
import matplotlib.pyplot as plt

REPO = "/content/P2Explore"
sys.path.insert(0, REPO)
from utils.FPUNet import FPUNet
device = "cuda" if torch.cuda.is_available() else "cpu"
FREE, OCC, UNK = 255, 0, 100   # FPUNet's pixel values
TARGET_RES, PATCH, STRIDE = 0.2, 128, 64

# which model to use
CKPT = ""   # leave empty: the new 20-epoch model if it exists, otherwise the authors' original; or paste a path to a .pth file
if not CKPT:
    new_model = "/content/drive/MyDrive/FPUNet_models/w06_e20_loss/best_fpunet_128_20.pth"
    CKPT = new_model if os.path.exists(new_model) else f"{REPO}/trained_model/best_fpunet_128.pth"
print("model:", CKPT)
if "YAML" not in globals():
    raise RuntimeError("Run 2a first and choose a .pgm and a .yaml file.")

# read the .yaml and its .pgm
info = {}
for line in open(YAML):
    if ":" in line and not line.strip().startswith("#"):
        k, v = line.split(":", 1); info[k.strip()] = v.strip()
img_path = info["image"] if os.path.isabs(info["image"]) else os.path.join(os.path.dirname(os.path.abspath(YAML)), info["image"])
if not os.path.exists(img_path):
    # Colab renames a file that is uploaded twice, e.g. "map (1).pgm"; use the uploaded .pgm instead
    pgms = [n for n in globals().get("uploaded", {}) if n.lower().endswith(".pgm")]
    if len(pgms) == 1:
        print(f"'{os.path.basename(img_path)}' not found, using the uploaded '{pgms[0]}' instead")
        img_path = pgms[0]
    else:
        raise FileNotFoundError("The map image (.pgm) was not uploaded. Run the upload cell again and select BOTH the .pgm and the .yaml.")
img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
if img.ndim == 3: img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
res = float(info["resolution"])
p = img / 255.0 if int(info.get("negate", 0)) else (255.0 - img) / 255.0
grid = np.full(img.shape, UNK, np.uint8)
grid[p > float(info.get("occupied_thresh", 0.65))] = OCC
grid[p < float(info.get("free_thresh", 0.25))] = FREE
grid[img == 205] = UNK   # map_saver's grey for unknown

# shrink to 0.2 m cells: a cell is a wall if any original cell in it is a wall
k = max(1, int(round(TARGET_RES / res)))
h, w = grid.shape; H, W = -(-h // k), -(-w // k)
pad = np.full((H * k, W * k), UNK, np.uint8); pad[:h, :w] = grid
blocks = pad.reshape(H, k, W, k).transpose(0, 2, 1, 3).reshape(H, W, k * k)
small = np.full((H, W), UNK, np.uint8)
small[(blocks == FREE).mean(-1) >= 0.5] = FREE
small[(blocks == OCC).any(-1)] = OCC

# run FPUNet on overlapping 128 x 128 pieces and stitch the answers
net = FPUNet(in_channels=3, n_classes=3, feature_scale=2)
net.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))
net = net.to(device).eval()
h, w = small.shape
HH = PATCH if h <= PATCH else -(-(h - PATCH) // STRIDE) * STRIDE + PATCH
WW = PATCH if w <= PATCH else -(-(w - PATCH) // STRIDE) * STRIDE + PATCH
big = np.full((HH, WW), UNK, np.uint8); big[:h, :w] = small
acc = np.zeros((3, HH, WW), np.float32); cnt = np.zeros((HH, WW), np.float32)
with torch.no_grad():
    for y in range(0, HH - PATCH + 1, STRIDE):
        for x in range(0, WW - PATCH + 1, STRIDE):
            piece = big[y:y + PATCH, x:x + PATCH]
            inp = np.stack([piece > 200, piece < 10, (piece > 90) & (piece < 110)]).astype(np.float32)
            acc[:, y:y + PATCH, x:x + PATCH] += net(torch.from_numpy(inp)[None].to(device))[0].cpu().numpy()
            cnt[y:y + PATCH, x:x + PATCH] += 1
idx = np.argmax(acc / cnt, axis=0)[:h, :w]
pred = np.full((h, w), FREE, np.uint8); pred[idx == 1] = OCC; pred[idx == 2] = UNK
pred[small == UNK] = UNK   # keep FPUNet's answer only inside the scanned area

erased = (small == OCC) & (pred == FREE)
print(f"map size: {img.shape[1] * res:.1f} m x {img.shape[0] * res:.1f} m -> {w} x {h} cells at 0.2 m")
print(f"black cells erased by FPUNet: {erased.sum()} of {np.sum(small == OCC)} ({np.mean(pred[small == OCC] == FREE):.1%})")
rgb = np.stack([pred] * 3, -1); rgb[erased] = (230, 40, 40)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(small, cmap="gray", vmin=0, vmax=255, interpolation="nearest"); ax[0].set_title("Your map (0.2 m cells)")
ax[1].imshow(rgb, interpolation="nearest"); ax[1].set_title("After FPUNet (red = erased)")
for a in ax: a.axis("off")
plt.show()
out = os.path.splitext(os.path.basename(YAML))[0] + "_fpunet_0.2m.png"
cv2.imwrite(out, pred); print("cleaned map saved as", out, "(Files panel on the left)")

## Part 3: Make training data and retrain FPUNet

- **3-0** adds our 15 extra buildings (ipa_room_segmentation) to the authors' 140. It should print **155 / 155**.
- **3a** drives a virtual robot through every building and saves "messy map → clean map" pairs (15–50 min).
- **3b** backs up those pairs to your Drive, so a disconnect doesn't mean redoing 3a.
- **3c** writes the training script: **20 epochs, wall weight 0.6, batch 32, best epoch picked by validation loss** (the same settings as our first retraining).
- **3d** trains and prints one line per epoch. Models are saved straight to your Drive: a copy every 5 epochs, the best epoch, the last epoch, and a checkpoint for resuming.

In [ ]:
# 3-0. Add our 15 extra training buildings: choose ipa_training_maps.zip
from google.colab import files
up = files.upload()
z = [n for n in up if n.endswith(".zip")][0]
%cd /content/P2Explore
!unzip -q -o "{z}" -d dataset
import os
n_fp, n_cs = len(os.listdir("dataset/floor_plans")), len(os.listdir("dataset/cluttered_scene"))
print("floor plans:", n_fp, "| with furniture:", n_cs)
print("OK: 140 + 15 extra buildings" if n_fp == n_cs == 155 else "CHECK: expected 155 and 155. Did you choose ipa_training_maps.zip?")

In [ ]:
# 3a. Generate the training data (uses the authors' 1sim_explore notebook). Takes 15-50 min.
# It prints one line per building (155 in total), then the number of pairs and the train / val / test split.
%cd /content/P2Explore
!jupyter nbconvert --to script 1sim_explore.ipynb --output sim_explore_script > /dev/null 2>&1
!python -u sim_explore_script.py 2>&1 | grep --line-buffered -E "Completed tasks|^[0-9]|Error|Traceback"
!ls -lh dataset/gen_data

In [ ]:
# 3b. Back up the training data to your Drive (about 0.5 GB), so a disconnect doesn't mean redoing 3a
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/FPUNet_models/gen_data
!cp /content/P2Explore/dataset/gen_data/train_data.npz /content/P2Explore/dataset/gen_data/val_data.npz /content/P2Explore/dataset/gen_data/test_data.npz /content/drive/MyDrive/FPUNet_models/gen_data/
!ls -lh /content/drive/MyDrive/FPUNet_models/gen_data

In [ ]:
%%writefile /content/P2Explore/train_script.py
# 3c. Writes the training script (running this cell does not start training; 3d does).
# Same settings as our first retraining: wall weight 0.6, batch 32, best epoch picked
# by validation loss. 20 epochs. Saves straight to Google Drive and continues from the
# last finished epoch if Colab disconnects.

import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data_utils
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm import tqdm
from utils.FPUNet import FPUNet

# ---------------- settings ----------------
num_epochs = 20
batch_size = 32
fig_size = 128
obs_weight = 0.6
class_weights = torch.tensor([0.2, obs_weight, 0.2])  # free occupied unknown
device = "cuda"

RUN_NAME = "w06_e20_loss"   # folder in your Drive; change it to start a brand-new run instead of resuming
DRIVE = "/content/drive/MyDrive"
SAVE_DIR = DRIVE + "/FPUNet_models/" + RUN_NAME
CKPT = SAVE_DIR + "/checkpoint.pth"

if not os.path.isdir(DRIVE):
    sys.exit("Google Drive is not connected: run the drive.mount cell first.")
os.makedirs(SAVE_DIR, exist_ok=True)


def get_model():
    return FPUNet(in_channels=3, n_classes=3, feature_scale=2).to(device)


# ---------------- data ----------------
file_name = "dataset/gen_data"
train_data = np.load(f"./{file_name}/train_data.npz", allow_pickle=True)
val_data = np.load(f"./{file_name}/val_data.npz", allow_pickle=True)
test_data = np.load(f"./{file_name}/test_data.npz", allow_pickle=True)


# each patch becomes 3 layers: free, occupied, unknown
def modify_data(data):
    res = torch.zeros((data.shape[0], 3, data.shape[2], data.shape[3]), dtype=torch.uint8)
    for i in range(data.shape[0]):
        res[i, 0] = data[i] > 200
        res[i, 1] = data[i] < 10
        res[i, 2] = (data[i] > 90) & (data[i] < 110)
    return res


def prepare(d, key):
    return modify_data(torch.from_numpy(d[key].reshape((-1, 1, fig_size, fig_size))))


input_train, out_train = prepare(train_data, "train_list"), prepare(train_data, "label_list")
input_val, out_val = prepare(val_data, "train_list"), prepare(val_data, "label_list")
input_test, out_test = prepare(test_data, "train_list"), prepare(test_data, "label_list")

train_loader = data_utils.DataLoader(data_utils.TensorDataset(input_train, out_train), batch_size=batch_size, shuffle=True)
val_loader = data_utils.DataLoader(data_utils.TensorDataset(input_val, out_val), batch_size=batch_size, shuffle=True)
test_loader = data_utils.DataLoader(data_utils.TensorDataset(input_test, out_test), batch_size=batch_size, shuffle=False)
print(f"patches: train {len(input_train)}, val {len(input_val)}, test {len(input_test)}", flush=True)

# ---------------- model, loss, optimizer (as in the authors' code) ----------------
torch.cuda.empty_cache()
net = get_model()
loss_func = nn.CrossEntropyLoss(weight=class_weights).to(device)
optimizer = optim.Adam(net.parameters(), lr=0.0002)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, eta_min=0.0001, T_max=10, last_epoch=-1)

state = {"epoch": 0, "min_loss_val": 1e9, "min_val_epoch": 0, "best_f1": 0.0, "best_f1_epoch": 0,
         "train_loss_list": [], "val_loss_list": [], "val_f1_list": []}

if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    net.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    state = ck["state"]
    print(f"Resuming: {state['epoch']} of {num_epochs} epochs already done "
          f"(to start a new run instead, change RUN_NAME in 3c)", flush=True)
else:
    print("Starting a new run, saving to", SAVE_DIR, flush=True)

# ---------------- training ----------------
for epoch in tqdm(range(state["epoch"], num_epochs), initial=state["epoch"], total=num_epochs):
    net.train()
    epoch_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        inputs, labels = inputs.to(torch.float32).to(device), labels.to(torch.float32).to(device)
        loss = loss_func(net(inputs), labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)

    # validation: loss, plus wall precision / recall / F1 (for information)
    net.eval()
    val_loss = 0
    tp, fp, fn = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(torch.float32).to(device), labels.to(torch.float32).to(device)
            outputs = net(inputs)
            val_loss += loss_func(outputs, labels).item()
            pred_wall = outputs.argmax(dim=1) == 1
            true_wall = labels[:, 1] > 0.5
            tp += (pred_wall & true_wall).sum().item()
            fp += (pred_wall & ~true_wall).sum().item()
            fn += (~pred_wall & true_wall).sum().item()
    val_loss /= len(val_loader)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    val_f1 = 2 * precision * recall / max(precision + recall, 1e-9)

    state["train_loss_list"].append(epoch_loss)
    state["val_loss_list"].append(val_loss)
    state["val_f1_list"].append(val_f1)

    # a copy every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save(net.state_dict(), f"{SAVE_DIR}/fpunet_128_epoch{epoch + 1}.pth")
    # MAIN best model: lowest validation loss (as in our first retraining)
    if val_loss < state["min_loss_val"]:
        torch.save(net.state_dict(), f"{SAVE_DIR}/best_fpunet_128_{num_epochs}.pth")
        state["min_loss_val"] = val_loss
        state["min_val_epoch"] = epoch + 1
    # extra copy: highest wall F1, only for comparison
    if val_f1 > state["best_f1"]:
        torch.save(net.state_dict(), f"{SAVE_DIR}/bestF1_fpunet_128_{num_epochs}.pth")
        state["best_f1"] = val_f1
        state["best_f1_epoch"] = epoch + 1

    line = (f"epoch {epoch + 1}: train_loss {epoch_loss:.4f} | val_loss {val_loss:.4f} | "
            f"wall precision {precision:.3f} recall {recall:.3f} F1 {val_f1:.3f}")
    print(line, flush=True)
    with open(SAVE_DIR + "/training_log.txt", "a") as log:
        log.write(line + "\n")

    # checkpoint for resuming (written to a temp file first, so a disconnect can't corrupt it)
    state["epoch"] = epoch + 1
    torch.save({"model": net.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "state": state}, CKPT + ".tmp")
    os.replace(CKPT + ".tmp", CKPT)

torch.save(net.state_dict(), f"{SAVE_DIR}/last_fpunet_128_{num_epochs}.pth")
print(f"best epoch by validation loss: {state['min_val_epoch']} (val_loss {state['min_loss_val']:.4f})")
print(f"best epoch by wall F1: {state['best_f1_epoch']} (F1 {state['best_f1']:.3f})")

# loss and F1 curves
x = np.arange(1, len(state["train_loss_list"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(x, state["train_loss_list"], label="Train loss")
ax[0].plot(x, state["val_loss_list"], label="Validation loss")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].legend(); ax[0].grid()
ax[1].plot(x, state["val_f1_list"], label="Validation wall F1")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Wall F1"); ax[1].legend(); ax[1].grid()
plt.tight_layout()
plt.savefig(SAVE_DIR + "/training_plot.png")
plt.close()

# ---------------- test the best (validation loss) model ----------------
def probability_to_label(origin):
    res = torch.ones(origin.shape[0], origin.shape[2], origin.shape[3], dtype=torch.int64)
    for i in range(origin.shape[0]):
        index_mat = torch.argmax(origin[i], dim=0)
        res[i][index_mat == 0] = 255
        res[i][index_mat == 1] = 0
        res[i][index_mat == 2] = 100
    return res


best_net = get_model()
best_net.load_state_dict(torch.load(f"{SAVE_DIR}/best_fpunet_128_{num_epochs}.pth", map_location=device, weights_only=True))
best_net.eval()
tp, fp, fn = 0, 0, 0
with torch.no_grad():
    for test_image, gt_img in test_loader:
        pred = probability_to_label(best_net(test_image.to(torch.float32).to(device)).cpu()).numpy()
        gt = probability_to_label(gt_img).numpy()
        tp += np.sum((gt == 0) & (pred == 0))
        fn += np.sum((gt == 0) & (pred != 0))
        fp += np.sum((gt != 0) & (pred == 0))
recall = tp / max(tp + fn, 1)
precision = tp / max(tp + fp, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-9)
print(f"test (best model) wall recall: {recall:.3f}, precision: {precision:.3f}, f1_score: {f1:.3f}")
print("all files are in", SAVE_DIR)

In [ ]:
# 3d. Train (about 5.5 min per epoch on a T4). Keep this tab open.
%cd /content/P2Explore
!python -u train_script.py 2>&1 | grep --line-buffered -v Warning

### If Colab disconnects during training
Your progress is saved in Drive. To continue:
1. Reconnect and run **Part 1** (1a to 1d).
2. Run **3R** below (instead of 3-0, 3a and 3b).
3. Run **3c** and **3d** again. It prints *"Resuming: X of 20 epochs already done"* and carries on.

To start a **brand-new run** instead of resuming, change `RUN_NAME` at the top of 3c (and the matching folder name in 2b and 5b).

In [ ]:
# 3R. ONLY after a disconnect: restore the training data from your Drive (instead of 3-0, 3a and 3b)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/P2Explore/dataset/gen_data
!cp /content/drive/MyDrive/FPUNet_models/gen_data/train_data.npz /content/drive/MyDrive/FPUNet_models/gen_data/val_data.npz /content/drive/MyDrive/FPUNet_models/gen_data/test_data.npz /content/P2Explore/dataset/gen_data/
!ls -lh /content/P2Explore/dataset/gen_data

## Part 4: Adding your own floor plans to the training data

Every training building needs **two PNG images with the same file name**:

| Folder | What it shows |
|---|---|
| `dataset/cluttered_scene/NAME.png` | the messy map (with furniture, or your raw SLAM map) |
| `dataset/floor_plans/NAME.png` | the clean version of the same map (the answer key) |

Pixel values (grayscale): **255 = free space, 0 = wall or furniture, 100 = outside the building / unknown.**
Scale: **1 pixel = 0.2 m**. Both images must line up pixel for pixel (same size, no cropping or rotating).

For our own scans: the SLAM Toolbox map goes in `cluttered_scene`, and a copy where you erased the furniture and noise (painted white) goes in `floor_plans`. Keep a few of your own buildings out of training to test on.

Upload the images into those folders (Files panel on the left) after 3-0, then run **3a, 3b, 3c and 3d**. Change `RUN_NAME` in 3c first, so it starts a new run instead of resuming the old one.

## Part 5: Compare the models

Run **5a** and choose **fpunet_test_maps.zip**. It holds 5 buildings that were **never used for training** (with answer keys) and 4 real robot maps.
**5b** scores every model it finds on exactly the same maps:

| Name in the table | Model |
|---|---|
| `original` | the authors' model (no retraining) |
| `first_10ep` | our first retraining: 10 epochs, wall weight 0.6 (only if you added `best_fpunet_128_10.pth`) |
| `new_20ep` | your 20-epoch run from 3d, best epoch by validation loss (**the main result**) |
| `new_F1` | your 20-epoch run, best epoch by wall F1 (for comparison) |

To include `first_10ep`, upload `best_fpunet_128_10.pth` into **P2Explore → trained_model** (Files panel), or put it in your Drive folder **FPUNet_models**.

**How to read it:** *furniture erased* should be high, *walls kept* should stay close to 99%, and *wall F1* should go up. "ALL TEST MAPS" shows the totals, with the **average** wall F1 in the last two columns.

In [ ]:
# 5a. Upload the test maps: choose fpunet_test_maps.zip
from google.colab import files
up = files.upload()
z = [n for n in up if n.endswith(".zip")][0]
!rm -rf /content/test_maps && mkdir -p /content/test_maps && unzip -q -o "{z}" -d /content/test_maps
!ls /content/test_maps/test /content/test_maps/real

In [ ]:
# 5b. Score the original and the retrained FPUNet on the same test maps
import glob, os, sys
import numpy as np, cv2, torch

REPO = "/content/P2Explore"
TEST_DIR = "/content/test_maps"
sys.path.insert(0, REPO)
from utils.FPUNet import FPUNet
device = "cuda" if torch.cuda.is_available() else "cpu"
FREE, OCC, UNK = 255, 0, 100
PATCH, STRIDE = 128, 64

from google.colab import drive
drive.mount("/content/drive")

DRIVE_MODELS = "/content/drive/MyDrive/FPUNet_models"
RUN_DIR = DRIVE_MODELS + "/w06_e20_loss"   # change this if you changed RUN_NAME in 3c
EXTRA = ""   # optional: paste the path of any other .pth file to add it to the table
candidates = [
    ("original", f"{REPO}/trained_model/best_fpunet_128.pth"),
    ("first_10ep", f"{REPO}/trained_model/best_fpunet_128_10.pth"),
    ("first_10ep", f"{DRIVE_MODELS}/best_fpunet_128_10.pth"),
    ("new_20ep", f"{RUN_DIR}/best_fpunet_128_20.pth"),
    ("new_F1", f"{RUN_DIR}/bestF1_fpunet_128_20.pth"),
    ("extra", EXTRA),
]
MODELS = {}
for name, path in candidates:
    if path and name not in MODELS and os.path.exists(path):
        MODELS[name] = path
for k, v in MODELS.items():
    print(f"{k:10s} model: {v}")
if "new_20ep" not in MODELS:
    print("(new_20ep not found: run 3d first, or check RUN_DIR)")


def load(path):
    net = FPUNet(in_channels=3, n_classes=3, feature_scale=2)
    net.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
    return net.to(device).eval()


def clean(small, net):
    h, w = small.shape
    HH = PATCH if h <= PATCH else -(-(h - PATCH) // STRIDE) * STRIDE + PATCH
    WW = PATCH if w <= PATCH else -(-(w - PATCH) // STRIDE) * STRIDE + PATCH
    big = np.full((HH, WW), UNK, np.uint8); big[:h, :w] = small
    acc = np.zeros((3, HH, WW), np.float32); cnt = np.zeros((HH, WW), np.float32)
    with torch.no_grad():
        for y in range(0, HH - PATCH + 1, STRIDE):
            for x in range(0, WW - PATCH + 1, STRIDE):
                piece = big[y:y + PATCH, x:x + PATCH]
                inp = np.stack([piece > 200, piece < 10, (piece > 90) & (piece < 110)]).astype(np.float32)
                acc[:, y:y + PATCH, x:x + PATCH] += net(torch.from_numpy(inp)[None].to(device))[0].cpu().numpy()
                cnt[y:y + PATCH, x:x + PATCH] += 1
    idx = np.argmax(acc / cnt, axis=0)[:h, :w]
    pred = np.full((h, w), FREE, np.uint8); pred[idx == 1] = OCC; pred[idx == 2] = UNK
    pred[small == UNK] = UNK   # keep the answer only inside the scanned area
    return pred


def wall_f1(gt, p):
    tp = np.sum((gt == OCC) & (p == OCC)); fn = np.sum((gt == OCC) & (p != OCC)); fp = np.sum((gt != OCC) & (p == OCC))
    pr = tp / (tp + fp) if tp + fp else 0; rc = tp / (tp + fn) if tp + fn else 0
    return 2 * pr * rc / (pr + rc) if pr + rc else 0


nets = {k: load(v) for k, v in MODELS.items()}

print("\nTEST MAPS WITH AN ANSWER KEY (never used for training)")
print(f"{'map':22s} {'model':10s} {'furniture erased':>16s} {'walls kept':>10s} {'wall F1 before':>14s} {'after':>6s}")
totals = {k: [0, 0, 0, 0] for k in nets}
f1_before, f1_after = [], {k: [] for k in nets}
for f in sorted(glob.glob(f"{TEST_DIR}/test/*_input.png")):
    name = os.path.basename(f).replace("_input.png", "")
    inp = cv2.imread(f, cv2.IMREAD_GRAYSCALE); gt = cv2.imread(f.replace("_input", "_answer"), cv2.IMREAD_GRAYSCALE)
    furn = (inp == OCC) & (gt == FREE); wall = (inp == OCC) & (gt == OCC); seen = inp != UNK
    f1_before.append(wall_f1(gt[seen], inp[seen]))
    for k, net in nets.items():
        p = clean(inp, net)
        fe, wk = np.sum(p[furn] == FREE), np.sum(p[wall] == OCC)
        t = totals[k]; t[0] += fe; t[1] += furn.sum(); t[2] += wk; t[3] += wall.sum()
        f1_after[k].append(wall_f1(gt[seen], p[seen]))
        print(f"{name:22s} {k:10s} {fe / furn.sum():16.1%} {wk / wall.sum():10.1%} "
              f"{wall_f1(gt[seen], inp[seen]):14.3f} {wall_f1(gt[seen], p[seen]):6.3f}")
for k, t in totals.items():
    print(f"{'ALL TEST MAPS':22s} {k:10s} {t[0] / t[1]:16.1%} {t[2] / t[3]:10.1%} "
          f"{np.mean(f1_before):14.3f} {np.mean(f1_after[k]):6.3f}")

print("\nREAL ROBOT MAPS (no answer key: share of black cells erased)")
for f in sorted(glob.glob(f"{TEST_DIR}/real/*_input.png")):
    name = os.path.basename(f).replace("_input.png", "")
    inp = cv2.imread(f, cv2.IMREAD_GRAYSCALE); occ = inp == OCC
    line = f"{name:22s}"
    for k, net in nets.items():
        p = clean(inp, net)
        line += f"  {k}: {np.mean(p[occ] == FREE):5.1%}"
    print(line)